# Chapter 8 · 에너지 연구설계

**연구질문(과제):** 에너지·환경 시계열 연구질문을 방법 선택 → 기준선 → 검증 → 연구설계서 초안으로 연결하라.

> 위에서 아래로 실행하면 끝까지 동작합니다. 마지막 셀은 실행 결과 숫자로 연구설계서 초안을 자동 작성합니다.

## 1. 핵심 개념

**연구설계는 질문 → 데이터 → 방법 → 검증의 일관성 문제다.** 아래 표의 네 열이 한 줄로 이어지지 않으면 설계에 결함이 있다.

| 질문 유형 | 대표 방법 | 식별/타당성의 핵심 | 흔한 반려 사유 |
|---|---|---|---|
| 예측 | ARIMA/SARIMA, ML, 사전학습 모델 | 시간순 외표본, 기준선, 누수 차단 | 약한 기준선, 누수, 단일 구간 |
| 인과(개입) | ITS, DID, SCM | 평행추세·동시사건 부재·간섭 없음 | 위약·사전추세 점검 부재 |
| 구조(패널) | FE, RE, TWFE | 시간불변 교란 통제, 군집 추론 | 내생성, 소수 군집 |
| 불확실성/위험 | GARCH 등 | 분포·레짐, 외표본 위험지표 | 단일 표본, 경제적 유의성 부재 |

이 노트북의 **방법 추천 함수는 교육용 규칙(휴리스틱)** 이며 검증된 의사결정 절차가 아니다. 설계자의 판단을 대체하지 않는다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("demand")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 데이터 감사 → 방법 선택 → 파이프라인 → 설계서

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose

MODE = "level"
series = (np.log(y).diff().dropna() * 100) if MODE == "logret" else y.astype(float)
series = series.rename("target")
print("분석 대상 계열:", "일별 로그수익률(%)" if MODE == "logret" else D["label"], "| 관측 수:", len(series))

profile = {"n": len(series), "freq": FREQ, "m": M}
profile["missing_share"] = float(series.isna().mean())
z = (series - series.median()) / (1.4826 * np.median(np.abs(series - series.median())) + 1e-12)
profile["outlier_share(|robust z|>4)"] = float((np.abs(z) > 4).mean())
x = series.dropna().to_numpy(float)
profile["adf_p"] = float(adfuller(x, autolag="AIC")[1]); profile["kpss_p"] = float(kpss(x, regression="c", nlags="auto")[1])
if M > 1 and 2 * M <= len(series) and M <= 400:
    dec = seasonal_decompose(series.astype(float), model="additive", period=M)
    r_ = dec.resid.dropna(); sr_ = (dec.seasonal + dec.resid).dropna()
    profile["seasonal_strength"] = max(0.0, 1 - float(r_.var()) / float(sr_.var()))
else:
    profile["seasonal_strength"] = float("nan")
h_ = len(x) // 2; profile["levene_p(전후반 분산)"] = float(stats.levene(x[:h_], x[h_:]).pvalue)
def sup_f_mean(v, trim=0.15, step=None):
    v = np.asarray(v, float); n = len(v); step = step or max(1, n // 100); best = (0.0, None)
    for k in range(int(trim * n), int((1 - trim) * n), step):
        a, b = v[:k], v[k:]; rss1 = np.sum((a - a.mean()) ** 2) + np.sum((b - b.mean()) ** 2); rss0 = np.sum((v - v.mean()) ** 2)
        F = (rss0 - rss1) / (rss1 / (n - 2)); best = max(best, (F, k), key=lambda t: t[0])
    return best
F_, k_ = sup_f_mean(x); profile["sup_F_mean_shift"] = float(F_)
print("\n데이터 프로파일:"); show(pd.Series(profile).to_frame("값"), 4)

In [ ]:
def recommend_methods(p, goal):
    # 교육용 규칙. 임계값은 관행적 값이며 연구마다 정당화가 필요하다.
    out, warn = [], []
    stationary = (p["adf_p"] < 0.05) and (p["kpss_p"] >= 0.05)
    seasonal = (not np.isnan(p["seasonal_strength"])) and p["seasonal_strength"] > 0.3
    if "예측" in goal or "수요" in goal or "건수" in goal or "매출" in goal:
        out.append("기준선: 나이브" + (" + 계절 나이브" if seasonal else ""))
        out.append("통계모형: " + ("SARIMA" if seasonal else "ARIMA") + (" (차분 필요)" if not stationary else ""))
        out.append("ML: 시차·달력 특성 + Ridge/부스팅 (시간순 CV)")
        if p["levene_p(전후반 분산)"] < 0.05: out.append("분산 불안정 -> 로그변환 또는 GARCH류 고려")
    if "정책" in goal or "효과" in goal or "DID" in goal or "ITS" in goal:
        out.append("개입 평가: ITS(단일 시계열) / DID(비교집단 있음) / SCM(처치 단위 1개, 도너 풀)")
        warn.append("인과 주장에는 평행추세·동시사건 부재·간섭 없음 점검과 위약검정이 필수")
    if p["n"] < 100: warn.append("관측 수가 적어 복잡한 모형은 과적합 위험")
    if p["sup_F_mean_shift"] > 50: warn.append("평균 이동(구조변화) 징후: 구간별 성능과 변화점 처리 필요")
    if p["outlier_share(|robust z|>4)"] > 0.01: warn.append("이상치 비율이 1% 초과: 처리 규칙을 사전에 정의")
    if not stationary: warn.append("정상성 검정이 정상과 부합하지 않음: 차분·추세 모형 비교")
    return out, warn
meta_goal = "수요 예측 / 규제 효과"
rec, warn = recommend_methods(profile, meta_goal)
print("연구 목적:", meta_goal); print("\n추천(교육용 규칙):"); [print(" -", r) for r in rec]
print("\n경고:"); [print(" -", w) for w in warn]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def dm_p(d):
    d = np.asarray(d, float); n = len(d)
    if n < 3 or np.std(d) == 0: return np.nan
    st = d.mean() / np.sqrt(np.var(d, ddof=1) / n); return float(2 * (1 - stats.t.cdf(abs(st), n - 1)))

lags = [1, 2, 3] + ([M] if 1 < M <= 60 else [])
X = pd.DataFrame({f"lag_{k}": series.shift(k) for k in lags})
for w in (5, 20): X[f"rm_{w}"] = series.shift(1).rolling(w).mean()
if FREQ in ("D", "B"): X["dow"] = series.index.dayofweek
data = pd.concat([X, series], axis=1).dropna(); XX, yy = data.drop(columns="target"), data["target"]
n = len(data); n_folds = 6; fold = int(max(10, min(90, n // 15))); start = n - n_folds * fold
rows = []
for k in range(n_folds):
    a, b = start + k * fold, start + (k + 1) * fold
    yte = yy.iloc[a:b].to_numpy(); r_ = {"fold": k + 1}
    r_["naive"] = rmse(yte, XX["lag_1"].iloc[a:b])
    if 1 < M <= 60: r_["seasonal naive"] = rmse(yte, XX[f"lag_{M}"].iloc[a:b])
    r_["mean(train)"] = rmse(yte, np.full(len(yte), yy.iloc[:a].mean()))
    for name, mdl in {"Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
                      "HistGB": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.06, random_state=SEED)}.items():
        mdl.fit(XX.iloc[:a], yy.iloc[:a]); r_[name] = rmse(yte, mdl.predict(XX.iloc[a:b]))
    rows.append(r_)
RES = pd.DataFrame(rows).set_index("fold")
print(f"롤링 오리진 {n_folds}폴드 × {fold}관측 (1단계 예측, 시차는 실제값)"); show(RES)
summary = RES.agg(["mean", "std", "max"]).T; show(summary)
bench = [c for c in ("naive", "seasonal naive", "mean(train)") if c in RES.columns]
best_bench = RES[bench].mean().idxmin(); best_model = RES[["Ridge", "HistGB"]].mean().idxmin()
dd = RES[best_bench] - RES[best_model]
print(f"\n최고 기준선={best_bench} | 최고 모형={best_model} | 평균 개선={dd.mean():.4f} | 개선 폴드 {int((dd>0).sum())}/{len(dd)} | 폴드 t검정 p≈{dm_p(dd):.3f} (폴드 수가 적어 근사)")

In [ ]:
question_text = "에너지·환경 시계열 연구질문을 방법 선택 → 기준선 → 검증 → 연구설계서 초안으로 연결하라."

threats = ['재분석·모델 값과 실측의 차이', '기온 예보 오차가 예측 성능에 미치는 영향(미래 기온을 안다고 가정하면 낙관)', '기상 체제변화·기후 추세', '규제와 동시 기상 충격']
bench_txt = "계절 나이브, 기온 포함 회귀"
imp = (RES[best_bench].mean() - RES[best_model].mean()) / RES[best_bench].mean() * 100
if abs(imp) < 2:
    claim = "기준선과 평균 RMSE 차이가 %.1f%%로 미미해 개선을 주장하기 어렵다." % imp
elif imp > 0:
    claim = "기준선 대비 평균 RMSE가 %.1f%% 낮았으나 폴드 수가 적어 우열은 잠정적이다." % imp
else:
    claim = "모형이 기준선을 이기지 못했다(평균 RMSE 변화 %.1f%%). 이 결과를 정직하게 보고하는 설계가 필요하다." % imp
draft = f"""
# 연구설계서 초안 (자동 생성 · 검토 후 직접 수정)

## Research Question
{question_text}

## Data
- 자료: {D['label']} | DATA_KIND: {DATA_KIND}
- 단위/주기: 일별 전력수요(합성, 기온 의존) / {FREQ} (계절 주기 M={M}) | 관측 {profile['n']}개
- 데이터 품질: 결측 {profile['missing_share']:.1%}, 이상치 {profile['outlier_share(|robust z|>4)']:.2%}

## Empirical Strategy
- 목적: {meta_goal}
- 모형 후보: {', '.join(rec[:3])}
- 벤치마크: {bench_txt}
- 검증: 롤링 오리진 {n_folds}폴드, 폴드 크기 {fold}, 누수 차단(시차·롤링은 shift, 학습구간 통계로만 스케일)

## Preliminary Result (이 노트북 실행값)
- 최고 기준선 {best_bench}, 최고 모형 {best_model}. {claim}

## Robustness (예정)
- 학습 창·시차 집합 변경, 하위기간/하위집단, 대체 지표(MAE·MASE), 사전 지정 위약검정(해당 시)

## Main Risks (리뷰어가 공격할 지점)
""" + "\n".join(f"- {t}" for t in threats) + """

## Expected Contribution (보수적 서술)
- 위 결과가 기준선 대비 일관되게 개선될 때에 한해, 특정 자료·기간에서의 예측 개선을 보고한다. 일반화 주장은 하지 않는다.
"""
print(draft)

## 3. 제출 전 점검표
- [ ] 연구질문이 한 문장이며 방법·종속변수·해석 척도가 한 줄로 이어지는가
- [ ] 데이터 출처·라이선스·`DATA_KIND`(실제/합성)를 명시했는가
- [ ] 가장 강한 단순 기준선과 비교했는가 (약한 기준선 금지)
- [ ] 누수 경로(전처리, 미래 특성, 분할)를 하나씩 점검했는가
- [ ] 폴드/구간별 분포를 보고했는가 (평균만 보고하지 않음)
- [ ] 인과 주장 시 식별 가정과 위약검정이 있는가
- [ ] 위 "Main Risks"에 대한 대응이 설계서에 있는가

**제출:** 자동 생성 초안을 본인 연구에 맞게 고친 연구설계서(2쪽 이내), 프로파일 표, 롤링 오리진 결과표, 한계 단락.